# D1a 텐서와 순전파 — 실습 (W1)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> Colab에서 **런타임 → 런타임 유형 변경 → GPU** 로 설정하면 빠릅니다(없어도 동작).
> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 텐서의 shape을 **예측하고 → 실행으로 확인**하는 습관이 생긴다
2. reshape·dim·브로드캐스팅으로 shape을 자유롭게 다룬다
3. `nn.Linear` = `x @ Wᵀ + b` 임을 직접 검산한다
4. **XOR 신경망의 순전파를 텐서로 구현**해 네 점을 완주한다

**7단계 멘탈모델 초점:** 데이터(표현) + 모델

> 💡 **이번 주 훈련법:** 각 셀을 실행하기 **전에** 주석의 shape을 가리고 결과를 먼저 예측해 보세요.

## Part A. 텐서 만들기 · shape · dtype
텐서 = numpy 배열 + GPU 연산 + 자동미분(다음 주). 만들면서 shape과 dtype을 확인합니다.

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈(Part E에서 사용)

a = torch.tensor([[1., 2.], [3., 4.]])                  # 리스트에서 직접 (소수점 → float32)
print('a shape:', a.shape, '| dtype:', a.dtype)         # (2,2), float32
z = torch.zeros(2, 3)                                   # 0으로 채운 (2,3)
n = torch.randn(2, 2)                                   # 표준정규 난수(가중치 초기화용)
s = torch.arange(0, 10, 2)                              # 0,2,4,6,8
print('z:', z.shape, '| n:', n.shape, '| s:', s)        # shape 확인

x_int = torch.tensor([1, 2])                            # 소수점 없음 → int64 (신경망 입력 불가!)
print('x_int dtype:', x_int.dtype)                      # int64
x_float = x_int.___()                                   # ✍️ 빈칸: float32로 바꾸는 메서드
print('x_float dtype:', x_float.dtype)                  # float32

device = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU 있으면 cuda (앞으로 매주 등장)
print('device:', device)                                # Colab GPU 설정 시 cuda

## Part B. 인덱싱 · 슬라이싱 · reshape
numpy와 동일. `-1`은 "나머지는 알아서 계산"— 한 곳에만 쓸 수 있습니다.

In [ ]:
m = torch.arange(12).reshape(3, 4)                      # 0~11을 (3,4)로
print(m)                                                # 행렬 확인
print('m[0]      :', m[0], '->', m[0].shape)            # 첫 행 (4,)
print('m[:, 1]   :', m[:, 1], '->', m[:, 1].shape)      # 둘째 열 (3,)
print('m[:2, 2:] :')                                    # 부분 행렬
print(m[:2, 2:], '->', m[:2, 2:].shape)                 # (2,2)

v = torch.arange(12)                                    # shape (12,)
r = v.reshape(___, 6)                                   # ✍️ 빈칸: "알아서 계산" 기호 → (2,6)
print('r shape:', r.shape)                              # torch.Size([2, 6])
flat = torch.rand(28, 28).reshape(1, -1)                # 이미지 펴기 미리보기(W3 Flatten)
print('28x28 ->', flat.shape)                           # (1, 784)

## Part C. dim(축) — 집계의 방향
`dim=k` = "**k번째 차원을 없앤다**". 실행 전에 결과 shape을 먼저 예측하세요.

In [ ]:
m2 = torch.tensor([[1., 2., 3.],
                   [4., 5., 6.]])                       # shape (2,3)
print('전체 합 :', m2.sum())                            # 21.
col_sum = m2.sum(dim=___)                               # ✍️ 빈칸: 행 차원을 없애 "열별 합"
print('열별 합 :', col_sum, col_sum.shape)              # [5.,7.,9.] / (3,)
row_sum = m2.sum(dim=1)                                 # 열 차원을 없애 "행별 합"
print('행별 합 :', row_sum, row_sum.shape)              # [6.,15.] / (2,)
print('행별 최대 위치:', m2.argmax(dim=1))              # 각 행의 최대 인덱스 → 분류 예측에 매주 사용

## Part D. 브로드캐스팅 — 규칙과 함정
뒤 차원부터 비교: 같거나 한쪽이 1이면 → 1인 쪽이 늘어남. **마지막 셀의 함정**을 꼭 확인하세요.

In [ ]:
A = torch.ones(2, 3)                                    # (2,3)
print('A + 10        ->', (A + 10).shape)               # 스칼라: (2,3)
bvec = torch.tensor([10., 20., 30.])                    # (3,)
print('A + (3,)      ->', (A + bvec).shape)             # 행마다 더해짐 (2,3) ← 편향 패턴
col = torch.ones(3, 1); row = torch.ones(1, 4)          # (3,1), (1,4)
print('(3,1)+(1,4)   ->', (col + row).shape)            # 양쪽 다 늘어남 (3,4)

u = torch.rand(100, 1); w_ = torch.rand(100)            # 함정 실험
print('(100,1)+(100,)->', (u + w_).shape)               # 에러가 아니라 (100,100)! shape 확인 습관

## Part E. 행렬곱과 nn.Linear 검산 ⭐
`(n,k) @ (k,m) → (n,m)`. `nn.Linear(in, out)`의 weight는 **(out, in)** — 그래서 전치가 필요합니다.

In [ ]:
torch.manual_seed(0)                                    # 재현성(시드 고정)
x = torch.randn(4, 3)                                   # 샘플 4개, 특성 3개
layer = nn.Linear(3, 2)                                 # 입력3 → 출력2 선형층(가중치는 난수 초기화)
print('weight:', layer.weight.shape)                    # (2,3) ← (출력, 입력) 순서!
print('bias  :', layer.bias.shape)                      # (2,)
manual = x @ layer.weight.___ + layer.bias              # ✍️ 빈칸: 전치(transpose) 속성
print('일치?', torch.allclose(manual, layer(x)))        # True → nn.Linear = x@Wᵀ+b 검산 완료
print('출력 shape:', layer(x).shape)                    # (4,2) — 배치 4개가 한 번에

## Part F. 뉴런 하나 손 계산 검증
설명서 §3의 손 계산(x=(2,3), w=(0.4,0.6), b=−1 → 1.6)을 코드로 확인합니다.

In [ ]:
x1 = torch.tensor([2., 3.])                             # 입력
w = torch.tensor([0.4, 0.6])                            # 가중치
b = torch.tensor(-1.)                                   # 편향
z = (w * x1).sum() + b                                  # 가중합 z = w·x + b
out = torch.relu(___)                                   # ✍️ 빈칸: 가중합을 ReLU에 통과
print('z =', round(z.item(), 4), '| ReLU(z) =', round(out.item(), 4))  # 1.6 / 1.6 — 손 계산과 일치

## Part G. XOR 신경망 순전파 완주 ⭐⭐
설명서 §10의 XOR 네트워크를 텐서로 구현합니다. 가중치는 손으로 정한 값 — **학습은 다음 주부터.**

- 은닉층: `h = ReLU(X @ W1 + b1)` · 출력층: `y = h @ w2`

In [ ]:
X4 = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # XOR 네 입력 (4,2)
target = torch.tensor([0., 1., 1., 0.])                 # XOR 정답
W1 = torch.tensor([[1., 1.], [1., 1.]])                 # 입력2 → 은닉2 가중치 (2,2)
b1 = torch.tensor([0., -1.])                            # 은닉 편향
w2 = torch.tensor([[1.], [-2.]])                        # 은닉2 → 출력1 가중치 (2,1)

h = torch.___(X4 @ W1 + b1)                             # ✍️ 빈칸: 은닉층 활성화 함수
y = h @ ___                                             # ✍️ 빈칸: 출력층 가중치와 행렬곱
print('은닉 h (새 좌표):')                              # 은닉 표현
print(h)                                                # (0,0),(1,0),(1,0),(2,1)
print('예측 y:', y.squeeze())                           # 0,1,1,0
print('정답  :', target)                                # XOR
print('전부 일치?', bool(torch.equal(y.squeeze(), target)))  # True면 완주 성공

In [ ]:
import matplotlib.pyplot as plt                         # 그래프

h1, h2 = h[:, 0], h[:, 1]                               # 은닉 공간 좌표
colors = ['tab:blue' if t == 0 else 'tab:red' for t in target]  # 클래스 0=파랑, 1=빨강
plt.figure(figsize=(5, 4))                              # 크기
plt.scatter(h1, h2, c=colors, s=140, edgecolors='k')    # 네 점(두 점은 (1,0)에서 겹침)
hx = torch.linspace(-0.2, 2.2, 50)                      # 경계선용 x 범위
plt.plot(hx, (hx - 0.5) / 2, 'g--', label='decision boundary')  # y=h1-2h2=0.5 경계
plt.xlabel('h1'); plt.ylabel('h2')                      # 축 라벨(영어)
plt.title('Hidden space: linearly separable now')       # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # 원래 공간에선 불가능했던 직선 분리!

> 원래 (x₁,x₂) 공간에서는 직선 하나로 못 나누던 네 점이, 은닉층이 만든 (h₁,h₂) 공간에서는 초록 점선 하나로 나뉩니다. **"은닉층 = 표현을 바꾼다"**의 실체입니다.

## Part H. 이미지도 텐서다
픽셀 = 숫자. 슬라이싱 = 크롭. (진짜 손글씨 데이터 MNIST는 W3에서 만납니다.)

In [ ]:
img = torch.linspace(0, 1, 28 * 28).reshape(28, 28)     # 28x28 그라데이션 "이미지"
img[7:21, 7:21] = 1.0                                   # 가운데 밝은 사각형 그리기(값 대입=그리기!)
crop = img[0:14, 0:14]                                  # 슬라이싱 = 크롭(왼쪽 위 1/4: 배경+사각형 모서리)
batch = img.unsqueeze(0)                                # 배치 차원 추가 → (1,28,28)
print('img:', img.shape, '| crop:', crop.shape, '| batch:', batch.shape)  # shape 확인

fig, axes = plt.subplots(1, 2, figsize=(7, 3))          # 두 장 나란히
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)        # 원본(밝기 기준 고정)
axes[0].set_title('image tensor (28,28)')               # 제목(영어)
axes[1].imshow(crop, cmap='gray', vmin=0, vmax=1)       # 크롭(같은 밝기 기준)
axes[1].set_title('crop = img[0:14, 0:14]')             # 제목(영어)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])                # 눈금만 제거(테두리는 남겨 경계 표시)
plt.show()                                              # 표시

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "(64, 3, 28, 28) 텐서의 각 숫자가 뭘 뜻하는지 물어봐 줘. 내가 먼저 답할게."
- "`m2.sum(dim=0)`과 `dim=1`의 결과 shape을 내가 예측할 테니 채점해 줘."
- "nn.Linear의 weight가 왜 (출력, 입력) 순서인지 설명해 줘."
- "XOR 표에서 (1,1) 입력의 h₂가 왜 1인지 검산해 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 텐서를 만들고 shape을 예측→확인했다 (reshape·dim·브로드캐스팅)
2. `nn.Linear` = `x @ Wᵀ + b` 를 직접 검산했다
3. XOR 신경망 순전파를 텐서로 완주하고, 은닉 공간에서 직선 분리를 확인했다

**스스로 점검**
- [ ] `m.sum(dim=0)`의 결과 shape을 실행 전에 맞힐 수 있다
- [ ] (100,1)+(100,)이 왜 (100,100)이 되는지 설명할 수 있다
- [ ] `nn.Linear(3,2)`의 weight가 (2,3)인 이유를 안다
- [ ] XOR 네트워크에서 h₂가 "둘 다 1인가?" 탐지기인 이유를 안다

**🔹심화 (선택)**
- W1·b1·w2를 바꿔 **AND / OR** 를 계산하는 네트워크를 만들어 보세요. (힌트: AND는 은닉층 없이 뉴런 1개로 충분 — 왜 그런지도 생각해 보기.)
- `torch.manual_seed`를 바꿔 가며 `nn.Linear` 초기 가중치가 달라지는 것을 확인해 보세요 — "학습 전 모델은 실행할 때마다 다른 엉터리"임을 체감.